In [ ]:
# During the class, we uploaded a PDF file containing
# information about Serenatto and its coffee beans.
# So, we started by creating a folder called “documents”
# in the Google Colab workspace and added the PDF file
# inside it. To upload the data,
#  we installed the LlamaIndex library:
!pip install -q llama-index

In [ ]:
# After installation, the next step was to load
# the PDF file containing the information about Serenatto.
# We used the SimpleDirectoryReader class to read the directory
# contents and load the document:declare SimpleDirectorReader.
from llama_index.core import SimpleDirectoryReader

In [ ]:
## directory of documents
documents = SimpleDirectoryReader(input_dir='documents')

In [ ]:
## input documents
documents.input_files

In [ ]:
# Com o diretório configurado, carregamos os dados do PDF.
# Isso nos permite acessar o conteúdo do documento e manipulá-lo:
docs = documents.load_data()

In [ ]:
# To ensure that the document was uploaded correctly,
# we check how many documents were processed.
# In the case of a single PDF, we expect the size to
# be equivalent to its number of pages:
docs

In [ ]:
## ten leafs
len(docs)

In [ ]:
# We view the metadata of the first page of the document,
# which contains information such as the file name,
# the path where it is stored, the size, creation date,
# and last modification date:
print(docs[0].get_metadata_str())

In [ ]:
# To better understand how the document is
# represented internally, we explore its
# structure using .__dict__:
docs[0].__dict__

In [ ]:
# To make processing and indexing easier, we split the document
# into smaller parts, called "chunks". We use the SentenceSplitter
# class to perform this splitting, setting a maximum size of 1200
# characters per chunk:
from llama_index.core.node_parser import SentenceSplitter

node_parser = SentenceSplitter(chunk_size=1200)

In [ ]:
# With the SentenceSplitter configured, we transform the document into nodes.
# Each node represents a chunk of the original text, which can be further
# processed and indexed individually:
nodes = node_parser.get_nodes_from_documents(docs, show_progress=True)

In [ ]:
# After splitting, we check how many nodes were generated.
# This helps us understand how the document was segmented
# and whether the chunk sizes are adequate:
len(nodes)

In [ ]:
# To understand how the chunks were created,
# we visualize the contents of the first node.
# This allows us to confirm that the text was
# divided correctly and that the information is preserved:
nodes[0]

In [ ]:
# Finally, we view the last node to ensure that
# the split was consistent across the document.
# This is important to avoid loss of information
# or improper fragmentation:
nodes[9]

In [ ]:
!pip install -q llama-index-embeddings-huggingface

In [ ]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

In [ ]:
# Classe personalizada para adaptar a assinatura esperada pelo Chroma
class ChromaEmbeddingWrapper:
    def __init__(self, model_name): # Inicializa o modelo de embeddings do Hugging Face com o nome especificado
        self.model = HuggingFaceEmbedding(model_name=model_name)

    def __call__(self, input): # Converte a entrada para um formato compatível com o HuggingFaceEmbedding
        return self.model.embed(input)


In [ ]:
# Definindo o modelo de embedding usado pelo chroma
embed_model_chroma = ChromaEmbeddingWrapper(model_name='intfloat/multilingual-e5-large')


In [ ]:
# Instalando o pacote para integrar o Chroma como um armazenamento de vetores no LlamaIndex
!pip install -q llama-index-vector-stores-chroma


In [ ]:
# Importando o ChromaDB
import chromadb

# Criando um cliente persistente do ChromaDB, armazenando os dados no diretório './chroma_db'
db = chromadb.PersistentClient(path='./chroma_db')


In [ ]:
# Importando o ChromaDB
import chromadb

# Criando um cliente persistente do ChromaDB, armazenando os dados no diretório './chroma_db'
db = chromadb.PersistentClient(path='./chroma_db')

# Atribuindo o cliente a uma variável para uso posterior
chroma_client = db

# Definindo o nome da coleção que será criada
collection_name = 'documentos_serenatto'

# Tenta obter uma coleção existente ou criar uma nova, caso não exista
try:
  chroma_collection = chroma_client.get_or_create_collection(
      name = collection_name,
      embedding_function = embed_model_chroma
  )

# Captura e exibe qualquer erro que ocorra durante a criação ou carregamento da coleção
except Exception as e:
  print(f'Erro ao carregar ou criar coleção: {e}')


In [ ]:
# Importando a classe para integrar o Chroma como um armazenamento de vetores no LlamaIndex
from llama_index.vector_stores.chroma import ChromaVectorStore

# Importando a classe que gerencia o armazenamento de índices e vetores
from llama_index.core import StorageContext


In [ ]:
# Criando uma instância do ChromaVectorStore para armazenamento de vetores
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)

# Configurando um contexto de armazenamento com o vector_store como armazenamento de vetores
storage_context = StorageContext.from_defaults(vector_store=vector_store)


In [ ]:
# Definindo o modelo de embedding
embed_model = HuggingFaceEmbedding(model_name='intfloat/multilingual-e5-large')


In [ ]:
# Importando a classe VectorStoreIndex do LlamaIndex, que permite criar um índice de armazenamento de vetores
from llama_index.core import VectorStoreIndex

# Criando um índice de armazenamento de vetores
index = VectorStoreIndex(nodes, storage_context=storage_context, embed_model=embed_model)


In [ ]:
# Importando a função do LlamaIndex, que permite carregar um índice previamente salvo
from llama_index.core import load_index_from_storage

# Carregando um índice existente a partir do contexto de armazenamento
index = load_index_from_storage(storage_context, embed_model=embed_model)


In [ ]:
from google.colab import userdata
GROQ_API = userdata.get('GROQ_API')

In [ ]:
!pip install -q llama-index-llms-groq

In [ ]:
from llama_index.llms.groq import Groq

llm = Groq(model='llama3-70b-8192', api_key=GROQ_API)

In [ ]:
query_engine = index.as_query_engine(similarity_top_k=2, llm=llm)


In [ ]:
query_engine.query("Quais grãos estão disponíveis?").response


In [ ]:
chat_engine = index.as_chat_engine(mode='context', llm=llm)


In [ ]:
pergunta = chat_engine.chat('Quais grãos estão disponíveis?').response
print(pergunta)


In [ ]:
pergunta = chat_engine.chat('Quais são os preços dos grãos?').response
print(pergunta)


In [ ]:
pergunta = chat_engine.chat('Você pode me dar mais detalhes sobre o catuai amarelo?').response
print(pergunta)


In [ ]:
pergunta = chat_engine.chat('Qual é o preço dele?').response
print(pergunta)


In [ ]:
chat_engine.chat_history
